# 项目一：公司日常运营指标分析

## 业务背景

本分析针对电商平台日常运营数据，通过 **流量→转化→复购→用户分层** 四层递进分析，
识别销售漏斗中的薄弱环节，量化客户价值分布，为运营决策提供数据支撑。

**核心指标体系：**
- 流量指标：PV（页面浏览量）、UV（独立访客数）
- 转化指标：付费率、各环节转化率
- 复购指标：复购用户数、复购率
- 客户价值：RFM 分层（Recency / Frequency / Monetary）

**数据源：** `user_behavior`（用户行为日志）+ `orders`（订单记录）

---
## 1. 环境准备与数据获取

In [ ]:
# ══════════════════════════════════════════════════════════
# 环境准备 & 数据库连接（支持 SQLite 和 MySQL 双模式）
# ══════════════════════════════════════════════════════════
#
# 切换到 MySQL: 把下面 DB_TYPE = 'sqlite' 改为 DB_TYPE = 'mysql'
# 前提: 先运行 mysql_setup.py 或 migrate_to_mysql.py 把数据导入 MySQL
# MySQL 连接: localhost / root / 123456 / ecommerce_analysis

# sqlite3: Python 内置的轻量数据库引擎，无需安装服务器
#          适合学习和本地分析，SQL 语法和 MySQL 完全一致
import sqlite3

# pandas: 数据分析的核心库，DataFrame = 内存中的表格
#         命名习惯: import pandas as pd → 后面用 pd.read_sql()、pd.DataFrame()
import pandas as pd

# numpy: 科学计算库，提供高效的数组运算
#         np.select() 向量化条件判断、np.log() 对数变换等
import numpy as np

# matplotlib: Python 最基础的绑图库
#             matplotlib.pyplot 提供类似 MATLAB 的绑图 API
import matplotlib.pyplot as plt
# mticker: 刻度格式化工具，比如把 1000000 显示为 "1,000,000"
import matplotlib.ticker as mticker

# seaborn: 基于 matplotlib 的高级可视化库
#          默认样式比 matplotlib 好看，sns.set_theme() 一键设置全局风格
import seaborn as sns

# datetime: 处理日期时间，datetime.now() 获取当前时间
from datetime import datetime

# ---- 全局样式设置 ----
# context='talk': 字体和线条粗细适合演讲/报告（比 'notebook' 大，比 'poster' 小）
# font='Microsoft YaHei': 指定中文字体，防止中文显示为方框
sns.set_theme(style='whitegrid', context='talk', font='Microsoft YaHei')
# 解决 matplotlib 中文字体下负号显示为方框的问题
plt.rcParams['axes.unicode_minus'] = False

# ── 数据库模式选择 ──────────────────────────────────────
# 改为 'mysql' 即可切换（前提: 先运行 mysql_setup.py）
DB_TYPE = 'sqlite'  # 可选: 'sqlite' 或 'mysql'

if DB_TYPE == 'sqlite':
    conn = sqlite3.connect('ecommerce_ops.db')
    # SQLite 中的表名
    T_USER_BEHAVIOR, T_ORDERS = 'user_behavior', 'orders'
    print('[SQLite] ecommerce_ops.db')
elif DB_TYPE == 'mysql':
    import pymysql
    conn = pymysql.connect(
        host='localhost', user='root', password='123456',
        database='ecommerce_analysis', charset='utf8mb4',
    )
    # MySQL 中的表名（项目一无后缀，直接使用原始表名）
    T_USER_BEHAVIOR, T_ORDERS = 'user_behavior', 'orders'
    print('[MySQL] ecommerce_analysis')

In [ ]:
# ============================================================
# 数据获取：从数据库读取用户行为和订单数据（表名根据 DB_TYPE 自动切换）
# ============================================================

# pd.read_sql(sql语句, 数据库连接)
# 作用: 把 SQL 查询结果直接变成 pandas DataFrame（内存中的表格）
# 使用 f-string 将表名变量注入 SQL，实现 SQLite/MySQL 双模式兼容

# ---- 读取用户行为表 ----
# SELECT * 表示读取所有列
user_behavior_df = pd.read_sql(f'SELECT * FROM {T_USER_BEHAVIOR}', conn)

# ---- 读取订单表 ----
orders_df = pd.read_sql(f'SELECT * FROM {T_ORDERS}', conn)

# ---- 类型转换 ----
# SQLite 默认把日期存为字符串，MySQL 可能返回 datetime 对象
# 统一用 pd.to_datetime() 转成 pandas 的 datetime 类型，兼容两种数据库
# 转换后才能用 .dt.date、.dt.days 等时间操作
user_behavior_df['create_time'] = pd.to_datetime(user_behavior_df['create_time'])
orders_df['create_time'] = pd.to_datetime(orders_df['create_time'])

# ---- 数据概览 ----
# len(df): DataFrame 的行数
# .value_counts(): 统计每个值出现的次数，默认按次数降序排列
print(f'用户行为数据: {len(user_behavior_df):,} 条')
print(f'订单数据:     {len(orders_df):,} 条')
print(f'\n行为类型分布:')
print(user_behavior_df['action'].value_counts())

---
## 2. 流量分析：日 PV / UV 趋势

PV（Page View）衡量整体流量规模，UV（Unique Visitor）衡量独立用户覆盖。
二者的比值（PV/UV）反映用户平均浏览深度，是衡量内容吸引力的关键指标。

In [ ]:
# ============================================================
# 流量分析：按日聚合 PV（页面浏览量）和 UV（独立访客数）
# ============================================================
# PV = Page View: 每一次页面浏览都计数，一个用户一天可以产生多次 PV
# UV = Unique Visitor: 一个用户一天只计 1 次，按 user_id 去重

pv_uv_df = (
    user_behavior_df
    # .dt.date: 从 datetime 中提取日期部分（去掉时分秒）
    # 比如 2025-10-01 14:30:00 → 2025-10-01
    # 这样同一天的所有行为就会被分到同一组
    .groupby(user_behavior_df['create_time'].dt.date)
    # agg() 同时计算多个聚合指标:
    #   pv = count('id'):   数总共有多少行（每条行为记录有一个唯一 id）
    #   uv = nunique('user_id'): 数有多少个不重复的用户
    .agg(pv=('id', 'count'), uv=('user_id', 'nunique'))
    # reset_index(): 把 groupby 的索引(user_id)变回普通列
    .reset_index()
    # rename(): 把列名 create_time 改成 date，表意更清晰
    .rename(columns={'create_time': 'date'})
)

# 计算日均值（所有天数的平均值）
avg_pv = pv_uv_df['pv'].mean()
avg_uv = pv_uv_df['uv'].mean()

print(f'日均 PV: {avg_pv:,.0f}')            # :, 表示千分位逗号，如 45,230
print(f'日均 UV: {avg_uv:,.0f}')
# PV/UV 比 = 平均每个用户浏览了多少个页面，衡量用户"逛得深不深"
print(f'PV/UV 比: {avg_pv/avg_uv:.1f}（平均每用户浏览页面数）')

In [ ]:
# PV/UV 趋势图
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pv_uv_df['date'], pv_uv_df['pv'], color='#2196F3', linewidth=2, label='PV (页面浏览量)')
ax.plot(pv_uv_df['date'], pv_uv_df['uv'], color='#FF5722', linewidth=2, label='UV (独立访客)')
ax.set_title('日 PV / UV 趋势', fontweight='bold', fontsize=14)
ax.legend(loc='upper right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('日期')
ax.set_ylabel('数量')
plt.tight_layout()
plt.show()

---
## 3. 付费率分析

付费率 = 当日付费用户数 / 当日活跃用户数。这是连接「流量」和「收入」的核心桥梁指标。

> **注意：** 付费用户从 `orders` 表中统计（状态为 paid/shipped/completed），
> 而非从 user_behavior 中统计 pay 行为，因为订单表才是收入的确权数据源。

In [ ]:
# ============================================================
# 付费率分析
# ============================================================
# 付费率 = 当天付费用户数 / 当天活跃用户数
# 注意: 付费用户从 orders 表取（确权数据），不是从 user_behavior 表取

# ---- 每日活跃用户数（从行为表统计） ----
# groupby + nunique: 按日期分组后，统计每天有多少不重复的用户
daily_users = (
    user_behavior_df
    .groupby(user_behavior_df['create_time'].dt.date)['user_id']
    .nunique()                          # 每天不重复用户数
    .reset_index(name='total_users')    # 给计数结果命名为 total_users
    .rename(columns={'create_time': 'date'})
)

# ---- 每日付费用户数（从订单表统计） ----
# .isin([...]): 筛选订单状态在列表中的行
# paid=已付款, shipped=已发货, completed=已完成 → 这些都是"有效订单"
# unpaid=未付款, cancelled=已取消 → 排除
paid = orders_df[orders_df['order_status'].isin(['paid', 'shipped', 'completed'])]
daily_paid = (
    paid.groupby(paid['create_time'].dt.date)['user_id']
    .nunique()
    .reset_index(name='paid_users')
    .rename(columns={'create_time': 'date'})
)

# ---- 合并计算付费率 ----
# merge(on='date', how='left'): 以左表(daily_users)为准，按 date 列关联
# how='left': 保留左表所有日期，即使某天没人付费也保留（付费人数填 0）
pay_rate_df = daily_users.merge(daily_paid, on='date', how='left')
# fillna(0): 把合并后为空的付费人数填为 0（有些天没人付费）
pay_rate_df['paid_users'] = pay_rate_df['paid_users'].fillna(0).astype(int)
# 付费率 = 付费人数 / 活跃人数
pay_rate_df['payment_rate'] = pay_rate_df['paid_users'] / pay_rate_df['total_users']

avg_pay_rate = pay_rate_df['payment_rate'].mean()
print(f'平均日付费率: {avg_pay_rate:.2%}')
print(f'最高付费率:   {pay_rate_df["payment_rate"].max():.2%}')
print(f'最低付费率:   {pay_rate_df["payment_rate"].min():.2%}')

In [ ]:
# 付费率趋势图
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pay_rate_df['date'], pay_rate_df['payment_rate'],
        color='#4CAF50', linewidth=2, marker='o', markersize=4)
ax.axhline(avg_pay_rate, color='red', linestyle='--', alpha=0.7,
           label=f'均值 ({avg_pay_rate:.1%})')
ax.set_title('日付费率趋势', fontweight='bold', fontsize=14)
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('日期')
ax.set_ylabel('付费率')
plt.tight_layout()
plt.show()

---
## 4. 复购行为分析

复购用户定义为下单次数 ≥ 2 的付费用户。复购率 = 复购用户数 / 总付费用户数。
高复购率说明产品/服务体验好，用户愿意反复消费。

In [ ]:
# ============================================================
# 复购行为分析
# ============================================================
# 复购用户: 下单次数 ≥ 2 的付费用户
# 复购率 = 复购用户数 / 总付费用户数

# 先筛选有效订单（排除未支付和已取消）
valid_orders = orders_df[orders_df['order_status'].isin(['paid', 'shipped', 'completed'])]

# 按 user_id 分组，统计每个用户的订单数和消费总额
repurchase_df = (
    valid_orders
    .groupby('user_id')
    .agg(
        order_count=('order_id', 'nunique'),  # nunique: 一个用户可能有多个订单，去重计数
        total_amount=('amount', 'sum'),        # sum: 该用户所有订单金额之和
    )
    .reset_index()
    # query() 相当于 SQL 的 HAVING: 只保留订单数 >= 2 的用户
    .query('order_count >= 2')
)

# 复购率 = 复购用户数 / 所有下过单的用户数
repurchase_rate = len(repurchase_df) / orders_df['user_id'].nunique()

print(f'复购用户数: {len(repurchase_df)}')
print(f'复购率:     {repurchase_rate:.2%}')
if len(repurchase_df) > 0:
    print(f'复购用户人均订单数: {repurchase_df["order_count"].mean():.1f}')
    print(f'复购用户人均消费额: {repurchase_df["total_amount"].mean():.0f} 元')

# 看看复购用户的订单数集中在什么范围
print(f'\n复购用户订单数分布:')
print(repurchase_df['order_count'].value_counts().sort_index())

---
## 5. 漏斗流失分析

将用户行为路径建模为四步漏斗：**浏览 → 加购 → 下单 → 支付**。
每个环节的「环节转化率」= 当前环节人数 / 上一环节人数，
用于精确定位转化瓶颈。

> **方法要点：** 使用 `pivot_table` 构建用户-行为矩阵，
> 将每个用户的行为二值化（有该行为=1），再按列求和得到各环节人数。

In [ ]:
# ============================================================
# 漏斗分析：构建用户-行为矩阵
# ============================================================
# 这是整个项目一最重要的一个技巧
#
# 原始数据是"长表"格式: 每个用户-行为一行
#   user_1  |  view          |  2025-10-01
#   user_1  |  add_to_cart   |  2025-10-01
#   user_1  |  view          |  2025-10-01  (同一天看了两个商品)
#
# 我们需要转成"宽表"格式: 每个用户一行，列是各种行为
#   user_id  |  view  |  add_to_cart  |  place_order  |  pay
#   user_1   |   1    |       1       |      0        |   0

user_action_matrix = (
    user_behavior_df
    # pivot_table: 数据透视，index=行，columns=列，values=计什么
    # aggfunc='size': 统计每个 (user_id, action) 组合出现了几次
    # fill_value=0: 没有该行为的填 0
    .pivot_table(index='user_id', columns='action', aggfunc='size', fill_value=0)
    # clip(upper=1): 关键一步！把大于 1 的值全部截断为 1
    # 因为一个用户可能一天看了 5 个商品，view=5
    # 但我们只关心"他有没有浏览行为"，有就是 1，不管几次
    # 这样每列求和 = 到达该环节的用户总数
    .clip(upper=1)
)

print('用户-行为矩阵（前 10 行，每个用户一行）:')
print(user_action_matrix.head(10))
print(f'\n矩阵维度: {user_action_matrix.shape[0]:,} 用户 × {user_action_matrix.shape[1]} 行为类型')
print(f'\n各列求和 = 各环节到达人数:')
print(user_action_matrix.sum().astype(int))

In [ ]:
# ============================================================
# 漏斗各环节转化率计算
# ============================================================
# 漏斗顺序: 浏览 → 加购 → 下单 → 支付
#
# 两种转化率的区别:
#   overall_conversion_rate: 以第一环节(浏览)为基准，看各环节还剩多少人
#     → 回答"100 个浏览的人，最后有多少人支付？"
#   step_conversion_rate: 以前一环节为基准，看相邻环节流失了多少
#     → 回答"100 个加购的人，有多少人真的下单了？"

funnel_steps = ['view', 'add_to_cart', 'place_order', 'pay']
step_labels = ['浏览', '加购', '下单', '支付']

funnel_data = []
for step in funnel_steps:
    # 如果用户-行为矩阵里有这一列，求和；没有就填 0
    count = int(user_action_matrix[step].sum()) if step in user_action_matrix.columns else 0
    funnel_data.append({
        'step': step,
        'step_cn': step_labels[len(funnel_data)],
        'user_count': count,
    })

funnel_df = pd.DataFrame(funnel_data)
# 整体转化率: 每个环节人数 / 第一个环节(浏览)人数
# .loc[0, 'user_count']: 取第 0 行(浏览)的 user_count
funnel_df['overall_conversion_rate'] = funnel_df['user_count'] / funnel_df.loc[0, 'user_count']
# 环节转化率: 每个环节人数 / 上一环节人数
# .shift(1): 把整列向下移一行，第一行变成 NaN（因为没有"上一环节"）
funnel_df['step_conversion_rate'] = funnel_df['user_count'] / funnel_df['user_count'].shift(1)

print('转化漏斗明细:')
print(f'{"环节":>8s}  {"用户数":>8s}  {"整体转化":>8s}  {"环节转化":>8s}')
print('-' * 50)
for _, row in funnel_df.iterrows():
    print(f'{row["step_cn"]:>8s}  {row["user_count"]:>8,}  '
          f'{row["overall_conversion_rate"]:>7.1%}  {row["step_conversion_rate"]:>7.1%}')

In [ ]:
# 漏斗可视化
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_r', len(funnel_df))
bars = ax.bar(range(len(funnel_df)), funnel_df['user_count'], color=colors, edgecolor='white', width=0.6)
ax.set_xticks(range(len(funnel_df)))
ax.set_xticklabels(funnel_df['step_cn'], fontsize=13)
ax.set_title('用户行为转化漏斗', fontweight='bold', fontsize=14)
ax.set_ylabel('用户数')

# 在柱子上标注环节转化率
for i, (_, row) in enumerate(funnel_df.iterrows()):
    y_offset = max(funnel_df['user_count']) * 0.02
    ax.text(i, row['user_count'] + y_offset,
            f"环节转化 {row['step_conversion_rate']:.1%}",
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333')

plt.tight_layout()
plt.show()

---
## 6. RFM 客户价值分层

RFM 模型从三个维度量化客户价值：
- **R (Recency)**：最近一次消费距今的天数 —— 越小越好
- **F (Frequency)**：消费频次 —— 越大越好
- **M (Monetary)**：消费金额 —— 越大越好

对每个维度使用 `pd.qcut` 五等分评分（R 倒序，F/M 正序），
然后加总得到 RFM 总分进行分层。

> **工程要点：** 当数据有大量重复值时，`pd.qcut` 会报 `ValueError`。
> 下面定义了 `safe_qcut` 函数，使用 `duplicates='drop'` 安全处理。

In [ ]:
# ============================================================
# RFM 模型 — 构建三维客户价值指标
# ============================================================
# R (Recency):  最近一次消费距今多少天（越小 = 越活跃越好）
# F (Frequency): 总共买了多少次（越多越好）
# M (Monetary):  总共花了多少钱（越多越好）
#
# 下面从订单表中聚合这三个指标

def safe_qcut(series, q, labels):
    """
    安全分箱: 解决 pd.qcut 处理大量重复值时的 ValueError
    详见学习指南 5.2 节的图解说明
    """
    try:
        # duplicates='drop': 如果分箱边界重复了，自动合并分箱
        result, bins = pd.qcut(series, q, labels=labels, retbins=True, duplicates='drop')
    except ValueError:
        # 如果还是失败，退化为等距分箱 pd.cut
        result, bins = pd.cut(series, bins=min(q, len(series.unique())),
                              labels=labels[:min(q, len(series.unique()))],
                              retbins=True, duplicates='drop')
    return result

# ---- 构建 RFM 宽表 ----
now = pd.Timestamp.now()  # 当前时间，用于计算 recency
rfm_df = (
    valid_orders           # 复用前面筛好的有效订单
    .groupby('user_id')
    .agg(
        # R: 最近一次购买时间 → 待会算距今多少天
        last_purchase=('create_time', 'max'),
        # F: 不重复的订单数
        frequency=('order_id', 'nunique'),
        # M: 累计消费金额
        monetary=('amount', 'sum'),
    )
    .reset_index()
)

# 计算 Recency: 当前时间 - 最后一次购买时间 = 距今多少天
# .dt.days: 从 Timedelta 对象中提取天数
rfm_df['recency'] = (now - rfm_df['last_purchase']).dt.days
rfm_df['frequency'] = rfm_df['frequency'].astype(int)
rfm_df['monetary'] = rfm_df['monetary'].astype(float)

# 查看 RFM 指标的分布
print(f'RFM 数据: {len(rfm_df)} 个用户')
print(f'\nR (距上次购买): 均值={rfm_df["recency"].mean():.0f}天  中位数={rfm_df["recency"].median():.0f}天')
print(f'F (购买频次):   均值={rfm_df["frequency"].mean():.1f}次  中位数={rfm_df["frequency"].median():.0f}次')
print(f'M (消费金额):   均值={rfm_df["monetary"].mean():.0f}元  中位数={rfm_df["monetary"].median():.0f}元')
# 注意: 如果均值 >> 中位数，说明有少数大客户拉高了平均值，数据是右偏分布

In [ ]:
# ============================================================
# RFM 评分与客户分层
# ============================================================
# 用 pd.qcut 把每个维度按 5 等分打分（1-5 分）
# R 维度倒序: 离上次购买越近（recency 越小）→ 分越高（越活跃）
# F/M 维度正序: 次数越多/金额越大 → 分越高
# 总分 = R + F + M，范围 3-15 分

# ---- 各维度打分 ----
# labels=[5,4,3,2,1]: R 越小 → 分越高（最近买的 = 5 分，很久没买 = 1 分）
rfm_df['r_score'] = safe_qcut(rfm_df['recency'], 5, labels=[5, 4, 3, 2, 1])
# labels=[1,2,3,4,5]: F 越大 → 分越高
rfm_df['f_score'] = safe_qcut(rfm_df['frequency'], 5, labels=[1, 2, 3, 4, 5])
rfm_df['m_score'] = safe_qcut(rfm_df['monetary'], 5, labels=[1, 2, 3, 4, 5])

# ---- 向量化分层（比逐行 apply 快 10-50 倍） ----
# .astype(int): 把 categorical 类型转成整数，才能相加
score = rfm_df['r_score'].astype(int) + rfm_df['f_score'].astype(int) + rfm_df['m_score'].astype(int)
# np.select: 从上到下匹配，第一匹配的生效
#   如果 score=15: 满足 >=12 → '高价值客户'（不会继续往下检查）
#   如果 score=7:  不满足 >=9 → 不满足 >=6 → 走 default → '低价值客户'
conditions = [score >= 12, score >= 9, score >= 6]
choices = ['高价值客户', '中高价值客户', '中价值客户']
rfm_df['segment'] = np.select(conditions, choices, default='低价值客户')

# ---- 分层汇总画像 ----
segment_stats = (
    rfm_df.groupby('segment')
    .agg(
        user_count=('user_id', 'count'),
        avg_recency=('recency', 'mean'),
        avg_frequency=('frequency', 'mean'),
        avg_monetary=('monetary', 'mean'),
    )
    .round(1)  # 保留一位小数
)

print('客户分层结果:')
for seg, row in segment_stats.iterrows():
    pct = row['user_count'] / len(rfm_df)
    print(f'  {seg}: {int(row["user_count"])}人 ({pct:.1%})  '
          f'R={row["avg_recency"]:.0f}天  F={row["avg_frequency"]:.1f}次  M={row["avg_monetary"]:.0f}元')

---
## 7. 综合可视化看板

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 图 1: PV/UV 趋势
ax = axes[0, 0]
ax.plot(pv_uv_df['date'], pv_uv_df['pv'], color='#2196F3', linewidth=2, label='PV')
ax.plot(pv_uv_df['date'], pv_uv_df['uv'], color='#FF5722', linewidth=2, label='UV')
ax.set_title('日 PV / UV 趋势', fontweight='bold')
ax.legend(loc='upper right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('日期'); ax.set_ylabel('数量')

# 图 2: 付费率趋势
ax = axes[0, 1]
ax.plot(pay_rate_df['date'], pay_rate_df['payment_rate'], color='#4CAF50', linewidth=2, marker='o')
ax.axhline(avg_pay_rate, color='red', linestyle='--', alpha=0.7, label=f'均值 ({avg_pay_rate:.1%})')
ax.set_title('付费率趋势', fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel('日期'); ax.set_ylabel('付费率')

# 图 3: 漏斗分析
ax = axes[1, 0]
colors = sns.color_palette('Blues_r', len(funnel_df))
bars = ax.bar(range(len(funnel_df)), funnel_df['user_count'], color=colors, edgecolor='white')
ax.set_xticks(range(len(funnel_df)))
ax.set_xticklabels(funnel_df['step_cn'], rotation=15)
ax.set_title('转化漏斗', fontweight='bold')
ax.set_ylabel('用户数')
for i, (_, row) in enumerate(funnel_df.iterrows()):
    ax.text(i, row['user_count'] + max(funnel_df['user_count']) * 0.02,
            f"{row['overall_conversion_rate']:.1%}", ha='center', fontsize=12, fontweight='bold')

# 图 4: 客户分层饼图
ax = axes[1, 1]
seg_counts = rfm_df['segment'].value_counts()
colors_pie = ['#1565C0', '#42A5F5', '#90CAF9', '#E3F2FD']
wedges, texts, autotexts = ax.pie(
    seg_counts.values, labels=seg_counts.index,
    autopct='%1.1f%%', colors=colors_pie,
    explode=(0.05, 0.02, 0, 0),
    textprops={'fontsize': 12}
)
ax.set_title('客户分层分布', fontweight='bold')

fig.suptitle('公司日常运营指标分析看板', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('日常运营指标分析.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. 关键发现与建议

根据以上分析，重点关注以下方面：

1. **漏斗薄弱环节**：定位环节转化率最低的步骤，优先优化
2. **客户结构健康度**：高价值客户占比是否合理？低价值客户是否过多？
3. **流量质量**：PV/UV 比可以反映用户粘性，结合付费率判断流量质量
4. **复购驱动**：分析复购用户的行为特征，指导拉新策略

In [ ]:
# 关闭数据库连接
conn.close()
print('分析完成！')